# <center> QC

## <center> Description

- env: squidpy_v06
- Gene and sample level QC

## <center> Set-up

#### **libs**

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc
from math import ceil
## for vst
from pydeseq2.dds import DeseqDataSet

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

## plotting
import marsilea as ma
import marsilea.plotter as mp
from matplotlib.colors import NoNorm # for marsilea dotplot
import plotnine as p9
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

#### **paths**

In [2]:
from pathlib import Path

if "snakemake" in globals():
    # in_metadata = Path(snakemake.input["in_metadata"])
    # out_metadata = Path(snakemake.output["out_metadata"])
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    ## input
    in_adata = ROOT / "output/preprocessing/objects/make_adata/adata.h5ad"
    ## output
    ### objects
    out_adata = ROOT / "output/preprocessing/objects/QC/adata.h5ad"
    out_adata.parent.mkdir(parents=True, exist_ok=True) # make sure the output folder exists
    out_metadata = ROOT / "output/preprocessing/objects/QC/metadata.tsv"
    out_counts = ROOT / "output/preprocessing/objects/QC/counts.tsv"
    ### plots
    out_plots = ROOT / "output/preprocessing/plots/QC"
    out_plots.mkdir(parents=True, exist_ok=True)

#### **reproducibility**

In [3]:
import random
import numpy as np
import torch
import scanpy.external as sce

SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## <center> Main

#### **Load data**

In [4]:
adata = sc.read_h5ad(in_adata)

adata

AnnData object with n_obs × n_vars = 66 × 30155
    obs: 'disease_group', 'QC_group', 'condition_group'
    var: 'genesymbol'

In [5]:
adata.obs[['disease_group', 'condition_group']].value_counts()

disease_group  condition_group
HR_case        IFNG_neg           20
               IFNG_pos           20
HR_control     IFNG_neg           13
               IFNG_pos           13
Name: count, dtype: int64

In [6]:
# make a category with all conditions

adata.obs['grp'] = adata.obs['disease_group'].astype(str) + '__' + adata.obs['condition_group'].astype(str)

## <center> Gene-level QC

In [7]:
# minimal gene filtering

## at least in 5 samples (less than half of the samples in the smallest group)
sc.pp.filter_genes(adata, min_cells=5)

adata

AnnData object with n_obs × n_vars = 66 × 30130
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp'
    var: 'genesymbol', 'n_cells'

In [8]:
# Number of genes to be kept

n_genes=len(dc.pp.filter_by_expr(
    adata=adata,
    min_total_count=10, # Minimum total count required per gene across all samples.
    group='grp', # group by this
    min_prop=0.25, # Minimum proportion of samples in the smallest group that express the gene.
    min_count=10, # Minimum count required per gene for at least some samples (I think for the value of the smallest group)
    # Makes it a bit more stringent if the size of the smallest group > large_n (it increases the number of samples where a gene has to be expressed)
    # deactivated because it is hard to track what's going on
    large_n=0, 
    inplace=False
))

print(f"Genes kept after filtering: {n_genes}")

Genes kept after filtering: 19557


In [9]:
# plot where the thresholds are

dc.pl.filter_by_expr(
    adata=adata,
    group="grp",
    min_count=10,
    min_total_count=10,
    large_n=0,
    min_prop=0.25,
)
plt.savefig(out_plots / "genes_01.png", bbox_inches="tight")
plt.savefig(out_plots / "genes_01.pdf", bbox_inches="tight")

plt.close()

In [10]:
# do the filtering

dc.pp.filter_by_expr(
    adata=adata,
    min_total_count=10, # Minimum total count required per gene across all samples.
    group='grp', # group by this
    min_prop=0.25, # Minimum proportion of samples in the smallest group that express the gene.
    min_count=10, # Minimum count required per gene for at least some samples (I think for the value of the smallest group)
    # Makes it a bit more stringent if the size of the smallest group > large_n (it increases the number of samples where a gene has to be expressed)
    # deactivated because it is hard to track what's going on
    large_n=0, 
    inplace=True
)

adata

AnnData object with n_obs × n_vars = 66 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp'
    var: 'genesymbol', 'n_cells'

## <center> Count preprocessing

In [11]:
# counts shoud be raw

np.max(adata.X)

967301.0

In [12]:
# since there are decimal points on some counts, round
## NOTE: this is due to the tool used to generate the count matrices. Some genes are not integers. The count matrix still contains raw counts.

adata.X.data = np.round(adata.X.data)
adata.layers['cts.raw']=adata.X.copy()

#### **vst counts**

In [13]:
# pyDESeq2 apparently does not work well with sparse matrices, so we need to densify before running DeseqDataSet

adata_dds = adata.copy()
adata_dds.X = adata_dds.X.toarray()

In [14]:
# apply vst to the counts

dds = DeseqDataSet(
    adata=adata_dds,
    design="~ 1",           # no design; we’re just transforming
    refit_cooks=True,
    n_cpus=1 # >1 may result in errors
)
## unsupervised transformation (does not account for design)
dds.vst(use_design=False)
## Extract VST values 
vst_mat = dds.layers["vst_counts"].copy()
## add to pdata
adata.layers['cts.vst']=vst_mat

Fit type used for VST : parametric
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.07 seconds.

Fitting dispersions...
... done in 19.04 seconds.



#### **normalized counts**

In [15]:
adata.layers["cts.deseq2_norm"] = dds.layers["normed_counts"].copy()
## also log-transform
adata.X = adata.layers["cts.deseq2_norm"].copy()
sc.pp.log1p(adata, copy=False)
adata.layers["cts.deseq2_norm_log"] = adata.X.copy()

#### **HVGs**

In [16]:
## identify top highly variable genes
adata.X = adata.layers['cts.deseq2_norm_log'].copy()
sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000) # expects log data

adata

AnnData object with n_obs × n_vars = 66 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    layers: 'cts.raw', 'cts.vst', 'cts.deseq2_norm', 'cts.deseq2_norm_log'

## <center> Unsupervised clustering

- PCA --> ANOVA check associated metadata variables
- UMAP --> Leiden clustering: Can we identify the group B samples? Do they separate?

#### **PCA**

In [17]:
sc.tl.pca(adata, layer='cts.vst', n_comps=20, use_highly_variable=True, svd_solver='arpack',#key_added='PCA_cts.vst'
)

#### **ANOVA associations**

In [18]:
# get associations

## only cols without NAs to avoid crashes
select_cols=adata.obs.apply(lambda col: col.isna().sum()==0)
select_cols=select_cols[select_cols].index.to_list()
dc.tl.rankby_obsm(adata, key='X_pca',obs_keys=select_cols)

In [19]:
adata.uns['rank_obsm']

,obsm,obs,stat,pval,padj
0,PC03,condition_group,195.963125,3.775562e-21,3.020450e-19
1,PC03,grp,67.801650,1.481074e-19,5.924297e-18
2,PC01,QC_group,119.466158,2.823702e-16,7.529872e-15
3,PC05,disease_group,28.641089,1.254425e-06,2.508850e-05
4,PC02,QC_group,19.840113,3.453411e-05,4.604548e-04
...,...,...,...,...,...
75,PC10,QC_group,0.352826,5.546116e-01,9.887381e-01
76,PC09,QC_group,0.006979,9.336843e-01,9.887381e-01
77,PC08,QC_group,0.209127,6.490011e-01,9.887381e-01
78,PC14,QC_group,0.170721,6.808525e-01,9.887381e-01


In [20]:
# plot ANOVA

## select a reduced number of the most important variables explaining PCs for plotting
topn=5 # topn vars explaining PCs
PCs=['PC01', 'PC02', 'PC03', 'PC04'] # top PCs
df=adata.uns['rank_obsm'].copy()
df=df[df['padj']<0.05]
select_vars=(
    ## filter for PCs
    df[df['obsm'].isin(PCs)].
    ## select topn by PC
    sort_values(by=['obsm', 'padj'], ascending=[True, True]).
    groupby('obsm').head(topn)['obs'].unique()
)
select_vars=list(select_vars)

## make plot
dc.pl.obsm(
    adata=adata,
    key='rank_obsm',
    names=[],
    return_fig=True,
    nvar=4,
    titles=['PC scores', 'Adjusted p-values'],
    figsize=(20, 10),
    save=f"{str(out_plots)}/anova.png"
)
plt.close()

In [21]:
# # add col with sample_id for PCA
# adata.obs['col_sampleID']=adata.obs.index
# adata.obs['col_sampleID']=adata.obs['col_sampleID'].astype('category')

In [22]:
# plot PCAs

for i, var in enumerate(select_vars):
    with mpl.rc_context({'figure.figsize': (10, 10)}):
        plt.figure()
        sc.pl.pca(
            adata, color=[var], annotate_var_explained=True, 
            components = ['1,2', '3,4'],
            ncols=2, show=False, size=400, wspace=0.3
        )
        plt.savefig(out_plots / f"pca_0{i}.png", bbox_inches="tight")
        plt.close()

... storing 'grp' as categorical


<Figure size 1000x1000 with 0 Axes>

<Figure size 1000x1000 with 0 Axes>

<Figure size 1000x1000 with 0 Axes>

<Figure size 1000x1000 with 0 Axes>

#### **UMAPs**

In [23]:
## compute the neighborhood graph
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20) 

In [24]:
## find Leiden communities to flag B group

for clust_resol in [0.05,0.1,0.15,0.2]:
    print(f"- Communities (leiden): Finding cell clusters at resolution: {clust_resol}")
    sc.tl.leiden(
        adata, key_added=f"leiden_{clust_resol}", resolution=clust_resol,
        random_state=SEED,
        flavor='igraph'
    )

- Communities (leiden): Finding cell clusters at resolution: 0.05
- Communities (leiden): Finding cell clusters at resolution: 0.1
- Communities (leiden): Finding cell clusters at resolution: 0.15
- Communities (leiden): Finding cell clusters at resolution: 0.2


In [25]:
## UMAP

sc.tl.umap(adata)

In [26]:
# plot umap

varlist=['condition_group', 'disease_group', 'QC_group', 'leiden_0.15']

sc.pl.umap(
    adata, color=varlist, ncols=2, frameon=True, show=False,
    size=150,add_outline=True,outline_width=(0.1, 0.0),
    )
 
plt.savefig(out_plots / "umap_01.png", format='png', dpi=600, bbox_inches="tight")
plt.savefig(out_plots / "umap_01.pdf", format='pdf', dpi=600, bbox_inches="tight")
plt.close()

## <center> other plots

- Heatmap with expression of PODO markers (Fig 5b)
- UMAP with unsupervised clustering + expression of some markers (Fig 5c)

#### **Expression of PODO markers**

In [27]:
# customize the marsilea heatmap legend (e.g., add a border around the colorbar)

import matplotlib as mpl
from matplotlib.patches import Rectangle
from legendkit import ColorArt

class BorderedColorArt(ColorArt):
    def _make_cbar_box(self):
        super()._make_cbar_box()

        # If there is a title, _final_pack is a VPacker: [title, canvas]
        # Otherwise _final_pack is the DrawingArea itself.
        if self.title is not None:
            canvas = self._final_pack.get_children()[1]
        else:
            canvas = self._final_pack

        canvas.add_artist(
            Rectangle(
                (0, 0),
                self.width,
                self.height,
                facecolor="none",
                edgecolor="black",
                linewidth=1.5,
                zorder=200,
            )
        )

In [28]:
# prepare data ------------
select_genes = [
    'APOL1', 'ACTN4', 'LAMB2', 'ITGB1', 'SYNPO', 'VEGFA', 'KIRREL1', 
    'COL4A5', 'ITGA3', 'LAMA5', 'SULF1', 'NPNT', 'CD2AP', 'TCF21', 
    'FAT1', 'PODXL', 'PLA2R1', 'MAFB']
ptab = pd.DataFrame(adata.layers['cts.deseq2_norm_log'].copy(), index=adata.obs.index,columns=adata.var['genesymbol'])
## select genes
ptab = ptab[select_genes]
ptab = ptab.T


# plot ---------------------
w,h=7,3
h = ma.Heatmap(
    ptab, linewidth=0.1,linecolor='black',cmap="magma",
    width=w,height=h,
    legend=False # set to false to make a custom one below
)

## custom heatmap colorbar (this line must go before h.add_legends())
heat = h._layer_plan[0]
h.custom_legend(
    lambda: BorderedColorArt(
        mappable=mpl.cm.ScalarMappable(norm=heat.norm, cmap=heat.cmap),
        title='gene\nexpression',
        tick_color="black",
        tick_width=1,
    ),
    name="heatmap_legend",
)

## dendrograms
#h.add_dendrogram("left",method='ward',show=False)
#h.add_dendrogram("top",method='ward',show=False)

## annotations
### axes
h.add_left(mp.Labels(ptab.index, text_props={"size": [8 for x in range(len(ptab.index))]}), pad=.1)
h.add_bottom(mp.Labels(ptab.columns, text_props={"size": [8 for x in range(len(ptab.columns))]}), pad=.05)
### column annotations
col_anno = adata.obs.loc[ptab.columns, ["condition_group", "disease_group", "QC_group"]]
pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
pal = {cat:color for cat,color in zip(adata.obs['condition_group'].cat.categories,pal)}
#pal = dict(zip(adata.obs["condition_group"].cat.categories,adata.uns["condition_group_colors"])) # colors saved in .uns
h.add_top(
    mp.Colors(
        col_anno["condition_group"],palette=pal,label="condition_group", #linewidth=1.1,linecolor="black"
    ),
    size=0.3,
    pad=0.05,
    name='condition_group'
)
h.add_legends()
## title
h.add_title(top=f"Expression of selected markers across samples")

h.render()

## add a border around heatmap
ax = h.get_main_ax()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.1)
    spine.set_zorder(20)
ax.set_axis_on()
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.patch.set_alpha(0)
## add border around column color label
anno_ax = h.get_ax("condition_group")
for spine in anno_ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.1)
    spine.set_edgecolor("black")
anno_ax.set_axis_on()
anno_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
anno_ax.patch.set_alpha(0)


## save
h.save(out_plots / "heatmap.png")
h.save(out_plots / "heatmap.pdf")

# plt.show()
plt.close()

#### **UMAPs**

In [29]:
## prepare data
ptab = (
    pd.DataFrame(
        adata.obsm["X_umap"],
        index=adata.obs_names,
        columns=["UMAP1", "UMAP2"],
    )
    .join(adata.obs)
    .reset_index(names="sample")
)
## rename condition
ptab['condition_group']=['IFNG-' if x=='IFNG_neg' else 'IFNG+' for x in ptab['condition_group']]
ptab['condition_group']=ptab['condition_group'].astype('category').cat.reorder_categories(['IFNG-','IFNG+'],ordered=True)

ptab.head()

,sample,UMAP1,UMAP2,disease_group,QC_group,condition_group,grp,leiden_0.05,leiden_0.1,leiden_0.15,leiden_0.2
0,SD1,14.097491,1.751583,HR_case,A,IFNG-,HR_case__IFNG_neg,0,0,0,0
1,SD3,15.661191,3.868177,HR_case,A,IFNG-,HR_case__IFNG_neg,0,0,0,0
2,SD5,14.426194,2.259023,HR_case,A,IFNG-,HR_case__IFNG_neg,0,0,0,0
3,SD7,14.871943,3.214072,HR_case,A,IFNG-,HR_case__IFNG_neg,0,0,0,0
4,SD9,13.561943,1.285898,HR_case,A,IFNG-,HR_case__IFNG_neg,0,0,0,0


In [30]:
# IFNG and case/ctrl

w,h=6,5

pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
pal = {cat:color for cat,color in zip(ptab['condition_group'].cat.categories,pal)}

p=(
    p9.ggplot(ptab, p9.aes(x='UMAP1', y='UMAP2')) +
    p9.geom_point(p9.aes(fill='condition_group',shape='disease_group'), size=4, color='black', stroke=0.25) +
    p9.scale_fill_manual(pal) +
    p9.scale_shape_manual({'HR_case':'*','HR_control':'o'}) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':5})) +
    p9.theme_classic() +
    p9.labs(title="Unsupervised clustering of samples\n") +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text=p9.element_text(size=11), 
        #axis_line=p9.element_line(size=1.2),
        figure_size=(w, h),
    ) 
)

p.save(out_plots / "umap_02.png", width=w, height=h, limitsize=False)
p.save(out_plots / "umap_02.pdf", width=w, height=h, limitsize=False)

plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 5 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/preprocessing/plots/QC/umap_02.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 5 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/preprocessing/plots/QC/umap_02.pdf


In [31]:
# IFNG and case/ctrl

w,h=6,5

pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
pal = {cat:color for cat,color in zip(ptab['condition_group'].cat.categories,pal)}

p=(
    p9.ggplot(ptab, p9.aes(x='UMAP1', y='UMAP2')) +
    p9.geom_point(p9.aes(fill='condition_group',shape='disease_group'), size=4, color='black', stroke=0.25) +
    p9.scale_fill_manual(pal) +
    p9.scale_shape_manual({'HR_case':'*','HR_control':'o'}) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':5})) +
    p9.theme_classic() +
    p9.labs(title="Unsupervised clustering of samples\n") +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text=p9.element_blank(), 
        axis_ticks=p9.element_blank(), 
        #axis_line=p9.element_line(size=1.2),
        figure_size=(w, h),
    ) 
)

p.save(out_plots / "umap_03.png", width=w, height=h, limitsize=False)
p.save(out_plots / "umap_03.pdf", width=w, height=h, limitsize=False)

plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 5 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/preprocessing/plots/QC/umap_03.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 5 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/preprocessing/plots/QC/umap_03.pdf


In [32]:
# expression of some markers

## change name to genesymbol for plot below
tmp=adata.copy()
tmp.var_names = tmp.var['genesymbol']

varlist=['PLA2R1', 'ITGA3', 'PODXL', 'OLFM3']

sc.pl.umap(
    tmp, color=varlist, ncols=2, frameon=True, show=False,
    size=150,add_outline=True,outline_width=(0.1, 0.0),
    )
 
plt.savefig(out_plots / "umap_04.png", format='png', dpi=600, bbox_inches="tight")
plt.savefig(out_plots / "umap_04.pdf", format='pdf', dpi=600, bbox_inches="tight")
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/anndata/_core/anndata.py:835: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM']

    Inferred to be: categorical



#### **Save**

In [33]:
adata.write(out_adata)

In [34]:
# metadata
metadata = adata.obs.copy()

# raw counts
counts = pd.DataFrame(adata.layers['cts.raw'].toarray(),index=adata.obs.index,columns=adata.var.index)
## transpose, this format is expected for DESeq2 for example
counts = counts.T
## cols same order as in metadata index
counts = counts[ metadata.index.to_list() ]

all(counts.columns == metadata.index)

True

In [35]:
## metadata
metadata.to_csv(out_metadata,sep='\t')
## counts
counts.to_csv(out_counts,sep='\t')